# **<font color="red">With Memory Chatbot-Redis Docker</font>**

In [3]:
import uuid
from typing import List
from pydantic import BaseModel, Field

from langchain_ollama import ChatOllama
from langchain_core.messages import SystemMessage
from langchain_core.runnables import RunnableConfig

from langgraph.graph import StateGraph, START, END, MessagesState
from langgraph.store.redis import RedisStore
from langgraph.store.base import BaseStore


# =====================================================
# 1. Initialize LLMs
# =====================================================

chat_llm = ChatOllama(
    model="llama3.2:3b",
    temperature=0.5,
)

memory_llm = ChatOllama(
    model="qwen3:1.7b",
    temperature=0.3,
)


# =====================================================
# 2. System Prompt
# =====================================================

SYSTEM_PROMPT_TEMPLATE = """You are a helpful assistant with memory capabilities.

If user-specific memory is available, use it to personalize your responses.

Always:
- Address the user by name if known.
- Use stored context naturally.
- Be friendly and tailored.
- Suggest 3 relevant follow-up questions at the end.

User Memory:
{user_details_content}
"""


# =====================================================
# 3. Pydantic Memory Models
# =====================================================

class MemoryItem(BaseModel):
    text: str = Field(description="Atomic user memory")
    is_new: bool = Field(description="True if new memory")


class MemoryDecision(BaseModel):
    should_write: bool
    memories: List[MemoryItem] = Field(default_factory=list)


memory_extractor = memory_llm.with_structured_output(MemoryDecision)


# =====================================================
# 4. Memory Extraction Prompt
# =====================================================

MEMORY_PROMPT = """You manage long-term user memory.

CURRENT MEMORY:
{user_details_content}

TASK:
- Extract stable long-term facts (identity, job, preferences, projects).
- Keep atomic short sentences.
- Mark is_new=True only if not already stored.
- No guessing.
- If nothing useful, return should_write=False.
"""


# =====================================================
# 5. Remember Node
# =====================================================

def remember_node(state: MessagesState, config: RunnableConfig, *, store: BaseStore):
    user_id = config["configurable"]["user_id"]
    namespace = ("user", user_id, "details")

    # Fetch existing memory
    items = list(store.search(namespace))
    existing_memory = "\n".join(
        item.value["data"] for item in items
    ) if items else "(empty)"

    last_user_message = state["messages"][-1].content

    decision: MemoryDecision = memory_extractor.invoke(
        [
            SystemMessage(content=MEMORY_PROMPT.format(
                user_details_content=existing_memory
            )),
            {"role": "user", "content": last_user_message}
        ]
    )

    if decision.should_write:
        for memory in decision.memories:
            if memory.is_new:
                store.put(
                    namespace,
                    str(uuid.uuid4()),
                    {"data": memory.text}
                )

    return {}


# =====================================================
# 6. Chat Node
# =====================================================

def chat_node(state: MessagesState, config: RunnableConfig, *, store: BaseStore):
    user_id = config["configurable"]["user_id"]
    namespace = ("user", user_id, "details")

    items = list(store.search(namespace))
    user_memory = "\n".join(
        item.value["data"] for item in items
    ) if items else "(empty)"

    system_message = SystemMessage(
        content=SYSTEM_PROMPT_TEMPLATE.format(
            user_details_content=user_memory
        )
    )

    response = chat_llm.invoke([system_message] + state["messages"])

    return {"messages": [response]}


# =====================================================
# 7. Build Graph
# =====================================================

builder = StateGraph(MessagesState)

builder.add_node("remember", remember_node)
builder.add_node("chat", chat_node)

builder.add_edge(START, "remember")
builder.add_edge("remember", "chat")
builder.add_edge("chat", END)


# =====================================================
# 8. Use RedisStore
# =====================================================

REDIS_URI = "redis://localhost:6379"

with RedisStore.from_conn_string(REDIS_URI) as store:

    # Run ONLY once for fresh Redis instance
    store.setup()

    graph = builder.compile(store=store)

    config = {"configurable": {"user_id": "u1"}}

    # Store some memory
    graph.invoke(
        {"messages": [{"role": "user", "content": "Hi, my name is Vikas"}]},
        config,
    )

    graph.invoke(
        {"messages": [{"role": "user", "content": "I am an AI Engineer"}]},
        config,
    )

    # Ask question
    response = graph.invoke(
        {"messages": [{"role": "user", "content": "Explain GenAI simply"}]},
        config,
    )

    print("\nAssistant Response:\n")
    print(response["messages"][-1].content)

    print("\n" + "-" * 20 + " Stored Memories (Redis) " + "-" * 20)

    for item in store.search(("user", "u1", "details")):
        print(item.value["data"])

    print("-" * 60)



Assistant Response:

Hello there, John!

GenAI stands for Generative Artificial Intelligence. It's a type of AI that can create new, original content on its own. Imagine having a super-smart machine that can write stories, generate images, or even code like a human.

Here are some examples of what GenAI can do:

* **Text Generation**: Write articles, blog posts, or even entire books.
* **Image Generation**: Create realistic pictures or videos from scratch.
* **Code Generation**: Develop new software or apps using pre-existing programming languages.

GenAI uses complex algorithms and machine learning techniques to analyze patterns in data and generate new content that's similar yet unique. It's like having a creative partner who can help you with writing, art, or coding projects!

Would you like to know more about how GenAI works or its potential applications?

Follow-up questions:

1. How do you think GenAI will change the way we create content in various industries?
2. Are there any 

In [4]:
# =========================
# 9. Check Memory Without Above Code
# =========================
from langgraph.store.redis import RedisStore

REDIS_URI = "redis://localhost:6379"

with RedisStore.from_conn_string(REDIS_URI) as store:
    ns = ("user", "u1", "details")
    items = store.search(ns)

    print("-"*20 + " " + "Stored Memories (Redis)" + " " + "-"*20)
    for it in items:
        print(it.value["data"])
    print("-"*70)


-------------------- Stored Memories (Redis) --------------------
GenAI is AI that creates new content like text, images, or code.
----------------------------------------------------------------------


## **<font color="blue">Chatbot</font>**

In [6]:
import uuid
from typing import List
from pydantic import BaseModel, Field
from langchain_ollama import ChatOllama
from langchain_core.messages import SystemMessage
from langchain_core.runnables import RunnableConfig
from langgraph.graph import StateGraph, START, END, MessagesState
from langgraph.store.redis import RedisStore
from langgraph.store.base import BaseStore

# =========================
# 1. Initialize LLMs
# =========================
chat_llm = ChatOllama(model="llama3.2:3b", temperature=0.5)
memory_llm = ChatOllama(model="qwen3:1.7b", temperature=0.3)

# =========================
# 2. System Prompt
# =========================
SYSTEM_PROMPT_TEMPLATE = """You are a helpful assistant with memory capabilities.

Use stored memory naturally and personalize responses. Always:
- Address the user by name if known
- Suggest 3 relevant follow-up questions at the end

User Memory:
{user_details_content}
"""

# =========================
# 3. Pydantic Memory Models
# =========================
class MemoryItem(BaseModel):
    text: str
    is_new: bool

class MemoryDecision(BaseModel):
    should_write: bool
    memories: List[MemoryItem] = Field(default_factory=list)

memory_extractor = memory_llm.with_structured_output(MemoryDecision)

# =========================
# 4. Memory Extraction Prompt
# =========================
MEMORY_PROMPT = """You manage long-term user memory.

CURRENT MEMORY:
{user_details_content}

TASK:
- Extract stable facts (identity, job, preferences, projects)
- Keep atomic short sentences
- Mark is_new=True only if not already stored
- No guessing
- If nothing useful, return should_write=False
"""

# =========================
# 5. Remember Node
# =========================
def remember_node(state: MessagesState, config: RunnableConfig, *, store: BaseStore):
    user_id = config["configurable"]["user_id"]
    ns = ("user", user_id, "details")

    items = list(store.search(ns))
    existing_memory = "\n".join(it.value["data"] for it in items) if items else "(empty)"
    last_message = state["messages"][-1].content

    decision: MemoryDecision = memory_extractor.invoke([
        SystemMessage(content=MEMORY_PROMPT.format(user_details_content=existing_memory)),
        {"role": "user", "content": last_message}
    ])

    if decision.should_write:
        for mem in decision.memories:
            if mem.is_new:
                store.put(ns, str(uuid.uuid4()), {"data": mem.text})

    return {}

# =========================
# 6. Chat Node
# =========================
def chat_node(state: MessagesState, config: RunnableConfig, *, store: BaseStore):
    user_id = config["configurable"]["user_id"]
    ns = ("user", user_id, "details")

    items = list(store.search(ns))
    user_memory = "\n".join(it.value["data"] for it in items) if items else "(empty)"

    system_message = SystemMessage(
        content=SYSTEM_PROMPT_TEMPLATE.format(user_details_content=user_memory)
    )

    response = chat_llm.invoke([system_message] + state["messages"])
    return {"messages": [response]}

# =========================
# 7. Build Graph
# =========================
builder = StateGraph(MessagesState)
builder.add_node("remember", remember_node)
builder.add_node("chat", chat_node)
builder.add_edge(START, "remember")
builder.add_edge("remember", "chat")
builder.add_edge("chat", END)

# =========================
# 8. Connect to Redis
# =========================
REDIS_URI = "redis://localhost:6379"

with RedisStore.from_conn_string(REDIS_URI) as store:
    store.setup()  # Run once if fresh Redis

    graph = builder.compile(store=store)
    config = {"configurable": {"user_id": "u1"}}

    print("Interactive chatbot started! Type 'exit' to quit.\n")

    while True:
        user_input = input("You: ").strip()
        if user_input.lower() in {"exit", "quit"}:
            print("Goodbye!")
            break

        response = graph.invoke({"messages": [{"role": "user", "content": user_input}]}, config)
        assistant_reply = response["messages"][-1].content
        print(f"\nAssistant:\n{assistant_reply}\n")

        # Show stored memories for debug (optional)
        print("-" * 20 + " Stored Memories " + "-" * 20)
        for item in store.search(("user", "u1", "details")):
            print(item.value["data"])
        print("-" * 60)

        print("\n"*2)
        print("="*80)



Interactive chatbot started! Type 'exit' to quit.



You:  Hi



Assistant:
Hello! It's nice to meet you. I see we're discussing GenAI, a fascinating topic. For those who may not know, GenAI refers to Generative Artificial Intelligence, which enables machines to create new and original content like text, images, or code.

I recall that our conversation started with your introduction. Would you like to dive deeper into the world of GenAI? What specific aspects would you like to explore?

Follow-up questions:
1. How do you think GenAI will impact various industries such as art, music, and writing?
2. Can GenAI be used for creative tasks, or is it primarily focused on generating data?
3. Are there any potential risks or concerns associated with the use of GenAI in content creation?

-------------------- Stored Memories --------------------
GenAI is AI that creates new content like text, images, or code.
------------------------------------------------------------





You:  My name is Vikas Prajapati and I am an AI Engineer in Lagozon Technology.



Assistant:
Nice to work with you, Vikas Prajapati! As an AI Engineer at Lagozon Technology, I'm sure you're always looking for ways to improve and expand the capabilities of GenAI.

GenAI is indeed a groundbreaking technology that enables the creation of new content, such as text, images, or code. Its potential applications are vast and varied, from generating creative writing to developing intelligent systems that can learn from data.

I recall that we previously discussed the use of GenAI in natural language processing (NLP) tasks, where it has shown impressive results in tasks like sentiment analysis and text classification. Have you explored any other areas where GenAI can be applied?

Here are three follow-up questions to further discuss:

1. How do you think GenAI can be used to improve the efficiency of software development processes?
2. What challenges have you encountered while working with GenAI, and how have you overcome them?
3. Are there any specific industries or domains

...


Assistant:
I'd be happy to talk about blockchain with you. Did you know that one of the key features of blockchain technology is its ability to record transactions in a decentralized manner? This means that instead of relying on a single central authority, like a bank or government, blockchain uses a network of computers around the world to verify and record transactions.

This creates a secure and transparent way for people to conduct transactions, such as sending cryptocurrency or tracking supply chain data. The use of cryptographic techniques, like hashing and digital signatures, ensures that the data is protected from tampering and unauthorized modifications.

For example, imagine you're using blockchain technology to track the origin of a product. The data would be recorded on a network of computers, each with a copy of the transaction history. This creates an immutable record that can be verified by anyone in the network.

What specific aspect of blockchain are you interested in

You:  Do you have remember my name?



Assistant:
I don't recall your name. You didn't mention it to me earlier. However, I can start fresh and ask: How can I assist you today? Would you like to discuss blockchain technology further or explore something else related to GenAI? 

Follow-up questions:

1. What specific aspect of blockchain would you like to know more about?
2. Are you looking to implement a blockchain solution in your current project or business?
3. Do you have any real-world examples of how blockchain is being used effectively in various industries?

-------------------- Stored Memories --------------------
GenAI is AI that creates new content like text, images, or code.
Blockchain is a decentralized ledger system that records transactions in a secure and transparent manner.
It uses cryptographic techniques to ensure data integrity and prevent unauthorized modifications.
Blocks in the chain are linked through hash values, making tampering difficult.
Consensus mechanisms like Proof of Work or Proof of Stake v

You:  exit


Goodbye!
